In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar


# 1. DATA CLEANING

# Agricultural transaction records in Agmarknet exceed 75M rows. To prevent 
# memory exhaustion and RAM thrashing, we stream data by year, project down to 
# required analytical coordinates, and apply early commodity filtering.


ZIP_PATH = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data\csv.zip"
TARGET_COMMODITY = "Tomato"
START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

COLUMNS_TO_KEEP = ["State", "District", "Market", "Commodity", "Variety", "Arrival_Date", "Modal_Price"]

def load_and_filter_commodity_panel(zip_path: str, commodity: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Extracts and filters longitudinal commodity transactions from a zipped CSV archive.
    Enforces standardized nomenclature across decentralized administrative reporting units.
    """
    chunks = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        csv_files = [f for f in z.namelist() if f.endswith('.csv')]
       
        
        for file in sorted(csv_files):
            # Stream read selected columns to maintain low peak memory overhead
            df_chunk = pd.read_csv(
                z.open(file),
                usecols=COLUMNS_TO_KEEP,
                dtype={
                    "State": "string",
                    "District": "string",
                    "Market": "string",
                    "Commodity": "string",
                    "Variety": "string",
                    "Modal_Price": "float64"
                }
            )
            
            # Standardize string fields to reconcile mandi-level reporting disparities
            df_chunk["Commodity"] = df_chunk["Commodity"].str.title().str.strip()
            matched = df_chunk[df_chunk["Commodity"] == commodity].copy()
            
            if not matched.empty:
                chunks.append(matched)
                
    df_all = pd.concat(chunks, ignore_index=True)
    
    # Parse longitudinal temporal identifiers
    df_all["Arrival_Date"] = pd.to_datetime(df_all["Arrival_Date"], format="%Y-%m-%d")
    
    # Restrict to target decadal analysis window
    temporal_mask = (df_all["Arrival_Date"] >= start_date) & (df_all["Arrival_Date"] <= end_date)
    df_filtered = df_all.loc[temporal_mask].copy()
    
    # Remove reporting anomalies (zero or negative prices recorded due to data-entry errors)
    df_filtered = df_filtered[df_filtered["Modal_Price"] > 0]
    
    # Standardize geographical identifiers
    df_filtered["State"] = df_filtered["State"].str.title().str.strip()
    df_filtered["Market"] = df_filtered["Market"].str.title().str.strip()
    
    return df_filtered

# Plotting tomato prices for different states, different varieties and different timelines




tomato_df['Variety'] = tomato_df['Variety'].astype(str).str.title().str.strip()

# Extract unique varieties and sort them
tomato_varieties = sorted(tomato_df['Variety'].unique().tolist())

print(f"Total unique tomato varieties recorded (2015-2025): {len(tomato_varieties)}\n")
print("List of Tomato Varieties:")
for variety in tomato_varieties:
    print(f"- {variety}")
    


# 3. Plot the 10-Year Price Trend

# Group by date and calculate the mean Modal_Price across all reporting markets
daily_avg = tomato_df.groupby('Arrival_Date')['Modal_Price'].mean().reset_index()

# Calculate a 30-day rolling average to smooth the trend line
daily_avg['Rolling_Price'] = daily_avg['Modal_Price'].rolling(window=30).mean()

# Initialize the plot
plt.figure(figsize=(14, 7))
# Plot the raw daily prices in the background (faded)
plt.plot(
    daily_avg['Arrival_Date'], 
    daily_avg['Modal_Price'], 
    color='lightblue', 
    alpha=0.5, 
    label='Daily Average Price'
)

# Plot the smoothed rolling average in the foreground
plt.plot(
    daily_avg['Arrival_Date'], 
    daily_avg['Rolling_Price'], 
    color='#2c3e50', 
    linewidth=2, 
    label='30-Day Rolling Average'
)

# Formatting the graph
plt.title('All-India Average Tomato Prices (2015 - 2025)', fontsize=16, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (INR per Quintal)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)

# Display the plot
plt.tight_layout()
plt.show()
for variety in varieties:
    # Filter the dataframe for the current variety in the loop
    var_df = tomato_df[tomato_df['Variety'] == variety]
    
    # Calculate the daily average across all mandis for this variety
    daily_avg = var_df.groupby('Arrival_Date')['Modal_Price'].mean().reset_index()
    
    # Safeguard: Skip varieties with insufficient data to plot a timeline
    if len(daily_avg) < 2:
        print(f"Skipping '{variety}': Not enough data points to plot.")
        continue
        
    # Calculate rolling average (min_periods=1 prevents NaN for sparse data)
    daily_avg['Rolling_Price'] = daily_avg['Modal_Price'].rolling(window=30, min_periods=1).mean()
    
    # Initialize a new figure for each variety
    plt.figure(figsize=(12, 5))
        # Plot raw daily prices (faded background)
    plt.plot(
        daily_avg['Arrival_Date'], 
        daily_avg['Modal_Price'], 
        color='lightblue', 
        alpha=0.5, 
        label='Daily Average'
    )
    
    # Plot smoothed rolling average (solid foreground)
    plt.plot(
        daily_avg['Arrival_Date'], 
        daily_avg['Rolling_Price'], 
        color='#2c3e50', 
        linewidth=2, 
        label='30-Day Rolling Avg'
    )
    
    # Format the individual graph
    plt.title(f'Price Trend: {variety} Tomato (2015 - 2025)', fontsize=14, pad=10)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price (INR / Quintal)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10)
    plt.tight_layout()
    
    # Display the plot before moving to the next variety in the loop
    plt.show()







# 2. Frequency domain estimators for fractional order integration tests

# Let X_t be a time series. Its spectral density f(lambda) satisfies:
#       f(lambda) ~ c * |lambda|^(-2d) as lambda -> 0
# The parameter d characterizes persistence:
#   - d = 0: Short memory (I(0)), autocorrelations decay exponentially.
#   - 0 < d < 0.5: Stationary long memory, autocorrelations decay hyperbolically.
#   - 0.5 <= d < 1: Non-stationary yet mean-reverting (infinite variance, shocks die out).
#   - d >= 1: Unit root non-stationary (permanent shock persistence).


def compute_demeaned_periodogram(series: np.ndarray):
    """
    Computes the sample periodogram I(lambda_j) using the Discrete Fourier Transform (DFT).
    
    Demeaning the series isolates intrinsic dynamic persistence from the deterministic
    mean, eliminating the zero-frequency spectral spike (lambda_0 = 0).
    """
    T = len(series)
    series_demeaned = series - np.mean(series)
    fft_coeffs = np.fft.fft(series_demeaned)
    
    # Fourier frequencies: lambda_j = 2 * pi * j / T, for j = 1, ..., floor(T/2)
    harmonic_freqs = 2.0 * np.pi * np.arange(1, T // 2 + 1) / T
    periodogram = (np.abs(fft_coeffs[1:T // 2 + 1]) ** 2) / (2.0 * np.pi * T)
    
    return harmonic_freqs, periodogram


def geweke_porter_hudak_estimator(series: np.ndarray, alpha: float = 0.6):
    """
    Geweke and Porter-Hudak (1983) log-periodogram semi-parametric regression estimator.
    
    Econometric Specification:
        ln(I(lambda_j)) = beta_0 - d * ln(4 * sin^2(lambda_j / 2)) + u_j
        
    Bandwidth Selection (m = T^alpha):
    Concentrating on m low-frequency Fourier ordinates captures long-memory dynamics
    around the origin while filtering out high-frequency seasonal ARMA components.
    """
    T = len(series)
    m = int(np.floor(T ** alpha))
    
    freqs, periodogram = compute_demeaned_periodogram(series)
    lambda_j = freqs[:m]
    I_j = periodogram[:m]
    
    # Harmonic transformation of the Fourier frequency vector
    X = -2.0 * np.log(2.0 * np.sin(lambda_j / 2.0))
    Y = np.log(I_j)
    
    # OLS estimation of memory parameter d
    X_dev = X - np.mean(X)
    Y_dev = Y - np.mean(Y)
    d_hat = np.sum(X_dev * Y_dev) / np.sum(X_dev ** 2)
    
    # Theoretical asymptotic variance under frequency-domain OLS: Var(u_j) = pi^2 / 6
    asymptotic_var_d = (np.pi ** 2 / 6.0) / np.sum(X_dev ** 2)
    se_d = np.sqrt(asymptotic_var_d)
    
    return d_hat, se_d, m


def robinson_local_whittle_estimator(series: np.ndarray, alpha: float = 0.6):
    """
    Robinson (1995) Gaussian Semi-Parametric Local Whittle Estimator.
    
    Minimizes the concentrated negative local Whittle log-likelihood objective:
        R(d) = ln( (1/m) * sum_{j=1}^m lambda_j^{2d} * I(lambda_j) ) - (2d/m) * sum_{j=1}^m ln(lambda_j)
        
    Efficiency Properties:
    The Local Whittle estimator achieves greater asymptotic efficiency than GPH:
        Var(d_LW) = 1 / (4m)  <  Var(d_GPH) = pi^2 / (24m) ~= 0.411 / m
    providing more powerful tests against the integer cointegration knife-edge.
    """
    T = len(series)
    m = int(np.floor(T ** alpha))
    
    freqs, periodogram = compute_demeaned_periodogram(series)
    lambda_j = freqs[:m]
    I_j = periodogram[:m]
    
    sum_log_lambda = np.sum(np.log(lambda_j))
    
    def concentrated_loss(d):
        scaled_I = (lambda_j ** (2.0 * d)) * I_j
        G_hat = np.mean(scaled_I)
        if G_hat <= 0 or np.isnan(G_hat):
            return 1e10
        return np.log(G_hat) - (2.0 * d / m) * sum_log_lambda

    # Bound optimization to standard theoretical boundaries (-0.49 to 1.49)
    result = minimize_scalar(concentrated_loss, bounds=(-0.49, 1.49), method="bounded")
    d_hat = result.x
    se_d = 1.0 / (2.0 * np.sqrt(m))
    
    return d_hat, se_d, m



# 3. Statistical Inference & Regime 

# Evaluating Wald test statistics against the two integer knife-edges:
#   1. Null Hypothesis of Short Memory: H0: d = 0
#   2. Null Hypothesis of Pure Unit Root: H0: d = 1


def classify_economic_regime(d_hat: float, se_d: float):
    """
    Conducts formal hypothesis testing and maps fractional memory parameters
    to structural economic behavior and cointegration model validity.
    """
    z_stat_zero = (d_hat - 0.0) / se_d
    z_stat_one = (d_hat - 1.0) / se_d
    
    p_val_zero = 2.0 * (1.0 - stats.norm.cdf(abs(z_stat_zero)))
    p_val_one = 2.0 * (1.0 - stats.norm.cdf(abs(z_stat_one)))
    
    reject_i0 = p_val_zero < 0.05
    reject_i1 = p_val_one < 0.05
    
    if d_hat <= 0:
        regime = "Short Memory / Noise [I(0)]"
        rationale = "Rapid geometric shock decay; standard stationary VAR/ARMA models valid."
    elif 0.0 < d_hat < 0.5:
        regime = "Stationary Long Memory [0 < d < 0.5]"
        rationale = "Covariance stationary; shocks decay hyperbolically. Differencing over-differences."
    elif 0.5 <= d_hat < 1.0:
        regime = "Non-Stationary Mean-Reverting [0.5 <= d < 1.0]"
        rationale = "Infinite variance, but shocks revert to mean. Rejects I(1); requires Fractional Cointegration (FCVAR)."
    else:
        regime = "Unit Root / Explosive [d >= 1.0]"
        rationale = "Permanent shock persistence; standard integer differencing / Johansen VECM valid."
        
    return {
        "Regime": regime,
        "Reject_I0": reject_i0,
        "Reject_I1": reject_i1,
        "P_Val_d0": p_val_zero,
        "P_Val_d1": p_val_one,
        "Rationale": rationale
    }

# 4. Aggregation and Long Memnory Panel Estimation

# We construct a balanced daily price series for each state:
#   1. Spatial Aggregation: Daily cross-mandi median P_st = Median_m(P_smt)
#      mitigates outlier contamination from illiquid rural collection centers.
#   2. Frequency Alignment: Resampling to a continuous calendar ('D') with time
#      interpolation resolves trading-calendar gaps (holidays, strikes, closures).
#   3. Logarithmic Scaling: ln(P_st) linearizes percentage changes and bounds variance.


def run_fractional_integration_pipeline(df: pd.DataFrame, min_obs: int = 500, bandwidth_alpha: float = 0.6) -> pd.DataFrame:
    """
    Executes the fractional integration diagnostic suite across all Indian states.
    """
    state_results = []
    unique_states = sorted(df["State"].unique())
    
    print(f"\n[Estimation] Initiating spectral diagnostics across {len(unique_states)} state panels...")
    
    for state in unique_states:
        sub_df = df[df["State"] == state]
        
        # Cross-mandi spatial median aggregation
        daily_median = sub_df.groupby("Arrival_Date")["Modal_Price"].median().sort_index()
        
        # Calendar regularization: reindex to complete calendar days and interpolate gaps
        daily_regularized = daily_median.asfreq("D")
        daily_filled = daily_regularized.interpolate(method="time").bfill().ffill()
        
        if len(daily_filled) < min_obs:
            print(f" - Skipping {state}: Insufficient temporal depth ({len(daily_filled)} < {min_obs}).")
            continue
            
        # Logarithmic transformation
        log_prices = np.log(daily_filled.values)
        T_eff = len(log_prices)
        
        # Frequency-domain parameter estimation
        d_gph, se_gph, _ = geweke_porter_hudak_estimator(log_prices, alpha=bandwidth_alpha)
        d_lw, se_lw, m_lw = robinson_local_whittle_estimator(log_prices, alpha=bandwidth_alpha)
        
        # Econometric hypothesis testing
        classification = classify_economic_regime(d_lw, se_lw)
        
        # 95% Asymptotic Confidence Bounds for Local Whittle
        ci_lower = d_lw - 1.96 * se_lw
        ci_upper = d_lw + 1.96 * se_lw
        
        state_results.append({
            "State": state,
            "Effective_T": T_eff,
            "Bandwidth_m": m_lw,
            "d_GPH": round(d_gph, 3),
            "d_LW": round(d_lw, 3),
            "SE_LW": round(se_lw, 3),
            "95%_CI_LW": f"[{round(ci_lower, 3)}, {round(ci_upper, 3)}]",
            "Reject_I(0)": classification["Reject_I0"],
            "Reject_I(1)": classification["Reject_I1"],
            "Regime": classification["Regime"]
        })
        
    summary_df = pd.DataFrame(state_results)
    return summary_df




if __name__ == "__main__":
    # If df or tomato_df is already present in your active Jupyter session, use it directly:
    if 'tomato_df' in locals():
        raw_data = tomato_df
    elif 'df' in locals():
        raw_data = df[df['Commodity'].astype(str).str.title().str.strip() == 'Tomato']
    else:
        raw_data = load_and_filter_commodity_panel(ZIP_PATH, TARGET_COMMODITY, START_DATE, END_DATE)
        
    diagnostic_table = run_fractional_integration_pipeline(raw_data, min_obs=500, bandwidth_alpha=0.6)
    
    # Format and display regression results
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.max_rows', None)
    
    print("\n" + "=" * 110)
    print("EMPIRICAL RESULTS: FRACTIONAL INTEGRATION ESTIMATION OF STATE-LEVEL TOMATO PRICE DYNAMICS")
    print("=" * 110)
    print(diagnostic_table[[
        "State", "Effective_T", "Bandwidth_m", "d_GPH", "d_LW", "SE_LW", 
        "95%_CI_LW", "Reject_I(0)", "Reject_I(1)", "Regime"
    ]].to_string(index=False))

In [ ]:
# Transforming the resolution of rainfall data
#Geocoding APMC mandis
# Merging the daily rainfall table with daily price data table



import pandas as pd
import zipfile
import os
import numpy as np
import xarray as xr
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# PART 1: QUERYING FOR A COMPLETE MASTER LIST OF MANDIS

zip_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data\csv.zip"

unique_chunks = []

with zipfile.ZipFile(zip_path, 'r') as z:
    csv_files = [f for f in z.namelist() if f.endswith('.csv')]
    print(f"Scanning all {len(csv_files)} files across the dataset for Andhra Pradesh mandis...\n")
    
    for file in sorted(csv_files):
        # We only read three columns to keep the memory footprint tiny
        temp_df = pd.read_csv(
            z.open(file),
            usecols=['State', 'District', 'Market'],
            dtype=str
        )
        
        mask = temp_df['State'].astype(str).str.contains('Andhra Pradesh', case=False, na=False)
        ap_subset = temp_df.loc[mask, ['State', 'District', 'Market']].dropna()
        
        # Clean up string formatting so we don't count the same mandi twice
        ap_subset['State'] = ap_subset['State'].str.strip().str.title()
        ap_subset['District'] = ap_subset['District'].str.strip().str.title()
        ap_subset['Market'] = ap_subset['Market'].str.strip().str.title()
        
        unique_chunks.append(ap_subset.drop_duplicates())

ap_mandis_df = pd.concat(unique_chunks, ignore_index=True).drop_duplicates()
ap_mandis_df = ap_mandis_df.sort_values(by=['District', 'Market']).reset_index(drop=True)

print("="*70)
print(f"TOTAL UNIQUE MANDIS FOUND IN ANDHRA PRADESH: {len(ap_mandis_df)}")
print("="*70)

output_file = "andhra_pradesh_mandis.csv"
ap_mandis_df.to_csv(output_file, index=False)
print(f"\nSaved complete list to '{output_file}'.")



# PART 2: MAPPING RAINFALL TO MANDIS 

# Prices are endogenous (driven by trader behavior), but rainfall is a pure, 
# exogenous shock. The issue is that rain doesn't just affect the exact GPS 
# coordinate of the market building—it affects the surrounding farms.
# 
# We use a spatial BallTree to draw 25km and 50km circles around each mandi:
# The 25km circle captures local farm-level weather shocks.
# The 50km circle captures broader district-level transport and supply shocks.

# 1. Load Geocoded Mandis

mandis = pd.read_csv("andhra_pradesh_mandis_geocoded.csv")
mandis["Latitude"] = pd.to_numeric(mandis["Latitude"], errors="coerce")
mandis["Longitude"] = pd.to_numeric(mandis["Longitude"], errors="coerce")
mandis = mandis.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)


# 2. Open NetCDF File

nc_path = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\tomato\rainfall data\RF25_ind2015_rfp25.nc"
ds = xr.open_dataset(nc_path)

lats = ds["LATITUDE"].values
lons = ds["LONGITUDE"].values

# 3. Build Spatial Index (BallTree)

lon_grid, lat_grid = np.meshgrid(lons, lats)
grid_coords_deg = np.column_stack([lat_grid.ravel(), lon_grid.ravel()])
grid_coords_rad = np.radians(grid_coords_deg)

earth_radius_km = 6371.0088
tree = BallTree(grid_coords_rad, metric="haversine")

mandi_coords_rad = np.radians(mandis[["Latitude", "Longitude"]].values)
radius_25km_rad = 25.0 / earth_radius_km
radius_50km_rad = 50.0 / earth_radius_km

indices_25km = tree.query_radius(mandi_coords_rad, r=radius_25km_rad)
indices_50km = tree.query_radius(mandi_coords_rad, r=radius_50km_rad)

# Fallback: If a mandi sits exactly between IMD grid points, just grab the nearest one
for i in range(len(mandis)):
    if len(indices_25km[i]) == 0:
        _, nearest = tree.query(mandi_coords_rad[i:i+1], k=1)
        indices_25km[i] = nearest[0]
    if len(indices_50km[i]) == 0:
        _, nearest = tree.query(mandi_coords_rad[i:i+1], k=1)
        indices_50km[i] = nearest[0]


# 4. Extract Daily Rainfall (Averaged Across Radii)

rain_cube = ds["RAINFALL"].values
rain_cube = np.where(rain_cube < 0, np.nan, rain_cube)

n_days = rain_cube.shape[0]
rain_flat = rain_cube.reshape(n_days, -1)

daily_rain_25km = np.empty((len(mandis), n_days), dtype=np.float32)
daily_rain_50km = np.empty((len(mandis), n_days), dtype=np.float32)

for m_idx in range(len(mandis)):
    daily_rain_25km[m_idx, :] = np.nanmean(rain_flat[:, indices_25km[m_idx]], axis=1)
    daily_rain_50km[m_idx, :] = np.nanmean(rain_flat[:, indices_50km[m_idx]], axis=1)


# 5. Final Table

dates = pd.to_datetime(ds["TIME"].values)
n_mandis = len(mandis)

final_df = mandis[["State", "District", "Market"]].loc[
    mandis.index.repeat(n_days)
].reset_index(drop=True)

final_df["date"] = np.tile(dates, n_mandis)
final_df["rainfall_25km_mm"] = daily_rain_25km.ravel()
final_df["rainfall_50km_mm"] = daily_rain_50km.ravel()

final_df = final_df.sort_values(by=["State", "District", "Market", "date"]).reset_index(drop=True)

output_filename = "andhra_pradesh_rainfall_2015_25km_50km.csv"
final_df.to_csv(output_filename, index=False)
print(f"Extraction successful! File saved as: {output_filename}")



# PART 3 & 4: EXTRACTING THE PRICE DATA IN CHUNKS

# The raw CSVs are massive. Loading them all at once crashes the RAM.
# Here, we read the data in 100,000-row chunks, keep only what we need for AP, 
# and throw the rest away before moving to the next chunk.


zip_folder = r"K:\RESEARCH PROJECTS\Prof. Gaurav Arora\price data"
zip_files = [f for f in os.listdir(zip_folder) if f.endswith('.zip')]
zip_path = os.path.join(zip_folder, zip_files[0])

rainfall_df = pd.read_csv("andhra_pradesh_rainfall_2015_25km_50km.csv")
rainfall_df["date"] = pd.to_datetime(rainfall_df["date"])
rainfall_df["clean_market"] = rainfall_df["Market"].astype(str).str.strip().str.lower()

print("Reading 2015 price data from zip...")
with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open("2015.csv") as f:
        df_2015 = pd.read_csv(f, low_memory=False)

cols = {c.strip().lower(): c for c in df_2015.columns}
state_col = next((cols[k] for k in cols if "state" in k), None)
market_col = next((cols[k] for k in cols if k in ["market", "mandi", "market_name"]), None)
comm_col = next((cols[k] for k in cols if "commodity" in k), None)
date_col = next((cols[k] for k in cols if k in ["arrival_date", "date", "reported_date"]), None)

price_col = next((cols[k] for k in cols if "modal" in k and "price" in k), None)
if not price_col:
    price_col = next((cols[k] for k in cols if "price" in k), None)

mask_ap = df_2015[state_col].astype(str).str.contains("Andhra", case=False, na=False)
mask_tomato = df_2015[comm_col].astype(str).str.contains("Tomato", case=False, na=False)

ap_tomatoes = df_2015[mask_ap & mask_tomato].copy()

ap_tomatoes["date"] = pd.to_datetime(ap_tomatoes[date_col], dayfirst=True, errors="coerce")
ap_tomatoes["clean_market"] = ap_tomatoes[market_col].astype(str).str.strip().str.lower()
ap_tomatoes["price_clean"] = pd.to_numeric(ap_tomatoes[price_col], errors="coerce")
ap_tomatoes = ap_tomatoes.dropna(subset=["date", "price_clean"]).reset_index(drop=True)

# Average out the different varieties (like Local vs Hybrid) to get one single daily price per mandi
daily_tomato_prices = (
    ap_tomatoes.groupby(["clean_market", "date"])["price_clean"]
    .mean()
    .round(2)
    .reset_index()
)
daily_tomato_prices.rename(columns={"price_clean": "tomato_avg_modal_price_rs_qtl"}, inplace=True)



# PART 5: BUILDING THE CONTINUOUS TIME SERIES PANEL
# This is a crucial step for the econometrics. Weather happens every day, even 
# when the mandi is closed for a weekend or a local strike. 
# 
# If we merged the rainfall data *onto* the price data, we would accidentally 
# delete the weather data for closed days. By merging the prices onto the full 
# 365-day rainfall calendar, we keep the true timeline intact.

final_panel = pd.merge(
    rainfall_df[["State", "District", "Market", "clean_market", "date", "rainfall_25km_mm", "rainfall_50km_mm"]],
    daily_tomato_prices,
    on=["clean_market", "date"],
    how="left"
)

final_panel.drop(columns=["clean_market"], inplace=True)
final_panel = final_panel.sort_values(by=["State", "District", "Market", "date"]).reset_index(drop=True)

out_filename = "ap_tomato_prices_and_rainfall_2015.csv"
final_panel.to_csv(out_filename, index=False)
print(f"\nFinished! File saved as: {out_filename}")



# PART 6: Plotting the data
# Before running any complex model, we just have a look at the data. 


df = pd.read_csv("ap_tomato_prices_and_rainfall_2015.csv")
df["date"] = pd.to_datetime(df["date"])

daily_avg = df.groupby("date").agg(
    avg_rainfall_25km=("rainfall_25km_mm", "mean"),
    avg_rainfall_50km=("rainfall_50km_mm", "mean"),
    avg_tomato_price=("tomato_avg_modal_price_rs_qtl", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6), dpi=150)

# Tomato Prices (Left Axis)
color_price = "#d62728" 
line1 = ax1.plot(
    daily_avg["date"], 
    daily_avg["avg_tomato_price"], 
    color=color_price, 
    linewidth=2, 
    label="Avg Tomato Price (₹/Quintal)"
)
ax1.set_xlabel("Date", fontsize=12, fontweight="bold")
ax1.set_ylabel("Modal Price (₹ / Quintal)", color=color_price, fontsize=12, fontweight="bold")
ax1.tick_params(axis="y", labelcolor=color_price)
ax1.grid(True, linestyle="--", alpha=0.3)

# Rainfall (Right Axis)
ax2 = ax1.twinx()
color_rain = "#1f77b4"
bar1 = ax2.bar(
    daily_avg["date"], 
    daily_avg["avg_rainfall_25km"], 
    color=color_rain, 
    alpha=0.45, 
    width=1.0, 
    label="Avg Rainfall (25km radius, mm)"
)
ax2.set_ylabel("Daily Rainfall (mm)", color=color_rain, fontsize=12, fontweight="bold")
ax2.tick_params(axis="y", labelcolor=color_rain)
ax2.set_ylim(0, daily_avg["avg_rainfall_25km"].max() * 2.5)

plt.title(
    "Andhra Pradesh (2015): State-Wide Daily Tomato Prices vs. Daily Rainfall",
    fontsize=14, 
    fontweight="bold", 
    pad=15
)

ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()

lines = line1 + [bar1]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig("ap_tomato_price_rainfall_2015.png", dpi=300)
plt.show()